# pHGFN — Results Analysis

pH-conditioned GFlowNet for RNA-targeted drug design. This notebook loads the
pipeline outputs and reproduces the headline figures/metrics.

**Scientific framing.** Selectivity is grounded in real 3D physics: GNINA docks
each molecule into the acidic KRAS i-motif and the neutral (unfolded) conformer.
A proxy oracle (frozen RNA-FM + ChemBERTa + trainable fusion) is trained on those
GNINA differentials and used as the fast in-loop reward; real GNINA re-validates
the top candidates.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.config import cfg
R = cfg.system.results_dir

## 1. GNINA label distribution (the supervision signal)

In [ ]:
labels = pd.read_csv(cfg.data.proxy_library_csv)
labels = labels[labels['ok'] == True]
gap = labels['acidic_score'] - labels['neutral_score']
print('molecules:', len(labels), '| frac preferring acidic:', round((gap>0).mean(),3))
plt.hist(gap, bins=40); plt.xlabel('acidic - neutral score'); plt.ylabel('count');
plt.title('GNINA selectivity gap'); plt.show()

## 2. Oracle (proxy) training curves

In [ ]:
from IPython.display import Image, display
for f in ['oracle_training_curves.png', 'gflownet_training_curves.png']:
    p = R / f
    if p.exists(): display(Image(str(p)))

## 3. Diversity metrics

In [ ]:
p = R / 'diversity_metrics.json'
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))

## 4. Pareto frontier (selectivity vs QED) + top candidates

In [ ]:
p = R / 'pareto_frontier.png'
if p.exists(): display(Image(str(p)))
pc = R / 'pareto_optimal.csv'
if pc.exists(): display(pd.read_csv(pc).head(10))

## 5. Real-GNINA validation of the top-K (proxy vs ground truth)

In [ ]:
p = R / 'candidates_gnina_validated.csv'
if p.exists():
    v = pd.read_csv(p); v = v[v['ok'] == True]
    print('proxy vs GNINA differential corr:', round(v['proxy_differential'].corr(v['gnina_differential']), 3))
    display(v.sort_values('gnina_differential', ascending=False).head(10))